# Kriteria 2 — Sistem RAG untuk Legal Team (UU & PP Ketenagakerjaan)

**Submission PGABL — Fepriyadi Harahap**

Asisten hukum yang menjawab pertanyaan Legal Team berdasarkan **empat dokumen wajib**:

| Dokumen | Tentang |
|---|---|
| UU Nomor 6 Tahun 2023 | Penetapan Perppu 2/2022 tentang Cipta Kerja menjadi Undang-Undang |
| PP Nomor 35 Tahun 2021 | PKWT, Alih Daya, Waktu Kerja & Waktu Istirahat, PHK |
| PP Nomor 51 Tahun 2023 | Perubahan atas PP 36/2021 tentang Pengupahan |
| PP Nomor 5 Tahun 2021 | Penyelenggaraan Perizinan Berusaha Berbasis Risiko |

Generator yang dipakai adalah **model hasil Kriteria 1** (SFT → GRPO) dari Hugging Face, sehingga jawaban
diawali proses berpikir `<think> ... </think>`.

```text
pertanyaan
    ├── metadata filter otomatis (mis. "PP 35/2021" -> peraturan_id = PP-35-2021)
    ├── HyDE: LLM membuat 2 jawaban hipotetis -> embedding gabungan
    ├── Ensemble Retriever (bobot 0.4 / 0.6)
    │     ├── BM25 (keyword) atas parent chunks
    │     └── Parent-Child Retriever: cari child chunk (Chroma) -> kembalikan parent chunk
    ├── Reranker cross-encoder -> Top-3 + relevance score
    │     ├── skor Top-1 >= threshold -> konteks dokumen lokal
    │     └── skor Top-1 <  threshold -> DuckDuckGo Search (internet)
    └── LLM fine-tuned: {context} + {question} -> <think> ... </think> jawaban + sitasi
```

> Jalankan di **Google Colab / Kaggle dengan GPU T4**. Keempat PDF diunduh otomatis dari
> [folder Google Drive](https://drive.google.com/drive/folders/1wGyyLRvuuJrroWb2TKmcBe_tY2c-huqY?usp=drive_link)
> menggunakan `gdown`.

## 0. Instalasi & Import

In [1]:
%%capture
!pip install -q unsloth
!pip install -q "langchain-classic==1.0.8" "langchain-community==0.4.2" "langchain-core==1.6.6"
!pip install -q "langchain-huggingface==1.2.2" "langchain-text-splitters==1.1.3" "chromadb==1.5.9"
!pip install -q "rank-bm25==0.2.2" "pymupdf==1.28.2" "ddgs==9.16.0" "sentence-transformers>=5.2.0" "gdown>=5.2.0"

In [2]:
# Unsloth di-import paling awal (sebelum transformers) untuk memuat model hasil fine-tuning.
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
import gc
import glob
import json
import os
import re
import time
import warnings
from collections import Counter
from typing import Any, Optional

import chromadb
import pandas as pd
import torch
from IPython.core.error import StdinNotImplementedError
from IPython.display import Markdown, display
from langchain_classic.retrievers import EnsembleRetriever, ParentDocumentRetriever
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import Chroma
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.retrievers import BaseRetriever
from langchain_core.stores import InMemoryStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 260)
os.environ["ANONYMIZED_TELEMETRY"] = "False"   # matikan telemetry chromadb

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

Device: cuda | Tesla T4


/tmp/ipykernel_23/2973466980.py:17: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


## 1. Memuat Dokumen PDF + Pembersihan Teks

PDF diunduh dari folder Google Drive ke `dokumen_uu_pp/` (dilewati bila sudah ada).

PDF UU/PP adalah hasil scan ber-OCR, sehingga setiap halaman membawa noise yang ikut ter-embed bila tidak dibuang:

```text
kop halaman      : PRESIDEN / REPUBLIK INDONESIA  (+ variasi OCR: REPUBUK, INDONESTA, FRESIDEN)
nomor halaman    : -21 -, 490 -, -L9-
footer           : www.peraturan.go.id, 2021, No.15, SK No 031803 A
penanda sambungan: "b. Perusahaan. . ." di kaki halaman (diulang di halaman berikutnya)
salah OCR        : (21 -> (2), danlatau -> dan/atau, 2O21 -> 2021
```

Halaman dengan < 100 karakter setelah dibersihkan dibuang. Pada PP 5/2021 ini adalah halaman
lampiran berupa tabel gambar tanpa teks.

In [4]:
DOC_CATALOG = {
    "UU Nomor 6 Tahun 2023.pdf": dict(
        peraturan_id="UU-6-2023", jenis="UU", nomor=6, tahun=2023, singkatan="UU 6/2023",
        judul="Penetapan Perppu Nomor 2 Tahun 2022 tentang Cipta Kerja menjadi Undang-Undang",
        topik="cipta kerja"),
    "PP Nomor 35 Tahun 2021.pdf": dict(
        peraturan_id="PP-35-2021", jenis="PP", nomor=35, tahun=2021, singkatan="PP 35/2021",
        judul="Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja",
        topik="ketenagakerjaan"),
    "PP Nomor 51 Tahun 2023.pdf": dict(
        peraturan_id="PP-51-2023", jenis="PP", nomor=51, tahun=2023, singkatan="PP 51/2023",
        judul="Perubahan atas Peraturan Pemerintah Nomor 36 Tahun 2021 tentang Pengupahan",
        topik="pengupahan"),
    "PP Nomor 5 Tahun 2021.pdf": dict(
        peraturan_id="PP-5-2021", jenis="PP", nomor=5, tahun=2021, singkatan="PP 5/2021",
        judul="Penyelenggaraan Perizinan Berusaha Berbasis Risiko",
        topik="perizinan berusaha"),
}


# Keempat PDF diunduh langsung dari folder Google Drive (akses: "Anyone with the link").
GDRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1wGyyLRvuuJrroWb2TKmcBe_tY2c-huqY?usp=drive_link"
DOC_DIR = "dokumen_uu_pp"


def missing_docs(folder):
    return [f for f in DOC_CATALOG if not os.path.exists(os.path.join(folder, f))]


if missing_docs(DOC_DIR):
    import gdown
    gdown.download_folder(url=GDRIVE_FOLDER_URL, output=DOC_DIR, quiet=False)

assert not missing_docs(DOC_DIR), f"PDF tidak ditemukan di folder Google Drive: {missing_docs(DOC_DIR)}"
for file_name in DOC_CATALOG:
    print(f"{file_name:32s} {os.path.getsize(os.path.join(DOC_DIR, file_name)) / 1024**2:6.1f} MB")


Retrieving folder contents


Processing file 1_MwGh_otQ6gCamLbuWnwfeKZjdaRiuaW PP Nomor 5 Tahun 2021.pdf
Processing file 15AxCnJyChz8X7jXvdeVwGS6WZZkJaVC5 PP Nomor 35 Tahun 2021.pdf
Processing file 1W--gzb_ERswssOC-ou0qdFvf_tYmgbQT PP Nomor 51 Tahun 2023.pdf
Processing file 1v3HML5sJAb1V2saKq6iG8bvwVS8hhoYB UU Nomor 6 Tahun 2023.pdf


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1_MwGh_otQ6gCamLbuWnwfeKZjdaRiuaW
To: /kaggle/working/dokumen_uu_pp/PP Nomor 5 Tahun 2021.pdf
100%|██████████| 17.1M/17.1M [00:00<00:00, 33.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=15AxCnJyChz8X7jXvdeVwGS6WZZkJaVC5
To: /kaggle/working/dokumen_uu_pp/PP Nomor 35 Tahun 2021.pdf
100%|██████████| 2.52M/2.52M [00:00<00:00, 210MB/s]
Downloading...
From: https://drive.google.com/uc?id=1W--gzb_ERswssOC-ou0qdFvf_tYmgbQT
To: /kaggle/working/dokumen_uu_pp/PP Nomor 51 Tahun 2023.pdf
100%|██████████| 2.77M/2.77M [00:00<00:00, 204MB/s]
Downloading...
From: https://drive.google.com/uc?id=1v3HML5sJAb1V2saKq6iG8bvwVS8hhoYB
To: /kaggle/working/dokumen_uu_pp/UU Nomor 6 Tahun 2023.pdf
100%|██████████| 85.4M/85.4M [00:01<00:00, 71.4MB/s]

UU Nomor 6 Tahun 2023.pdf          81.4 MB
PP Nomor 35 Tahun 2021.pdf          2.4 MB
PP Nomor 51 Tahun 2023.pdf          2.6 MB
PP Nomor 5 Tahun 2021.pdf          16.3 MB



Download completed


In [5]:
HEADER_RE = re.compile(r"(NDONES|PRES.?DEN|FRES.?DEN|REPU|PERATURAN\.GO\.ID|^SALINAN$|^SK NO|^\d{4}, NO\.\s?\d+$)")
PAGE_NO_RE = re.compile(r"^(-\s*.{1,5}\s*-|-?\s*\d{1,4}\s*-?)$")
CONTINUATION_RE = re.compile(r"(\.\s?\.\s?\.|…)$")
OCR_FIXES = [
    (re.compile(r"\((\d)1(?=[\s.;,])"), r"(\1)"),          # "(21 Upah" -> "(2) Upah"
    (re.compile(r"\b(dan|kabupaten)l(atau|kota)\b"), r"\1/\2"),
    (re.compile(r"(?<=\d)O|O(?=\d)"), "0"),                 # 2O21 -> 2021
]
PASAL_LINE_RE = re.compile(r"^Pasal (\d+[A-Z]?)$")
BAB_LINE_RE = re.compile(r"^BAB ([IVXLC]+)$")
STRUCT_START = re.compile(r"^(Pasal \d+[A-Z]?$|BAB [IVXLC]+$|Bagian |Paragraf |\(\d+\)|[a-z]\.\s|\d+\.\s)")


def clean_page(text):
    """Bersihkan satu halaman PDF. Return (teks_bersih, pasal_yang_dimulai, bab_yang_dimulai)."""
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    keep = []
    for i, line in enumerate(lines):
        if len(line) < 35 and HEADER_RE.search(line.upper()):
            continue                                   # kop + footer
        if i < 5 and PAGE_NO_RE.match(line):
            continue                                   # nomor halaman
        keep.append(line)
    if keep and CONTINUATION_RE.search(keep[-1]) and len(keep[-1]) < 40:
        keep = keep[:-1]                               # penanda sambungan halaman
    text = "\n".join(keep)
    for pattern, replacement in OCR_FIXES:
        text = pattern.sub(replacement, text)
    lines = text.split("\n")
    pasal = [m.group(1) for line in lines if (m := PASAL_LINE_RE.match(line))]
    bab = [m.group(1) for line in lines if (m := BAB_LINE_RE.match(line))]
    # gabungkan baris hasil word-wrap PDF; baris berpenanda struktur (Pasal, ayat, huruf) tetap baris baru
    merged = []
    for line in lines:
        if merged and not STRUCT_START.match(line) and not PASAL_LINE_RE.match(merged[-1]):
            merged[-1] = merged[-1] + " " + line
        else:
            merged.append(line)
    return re.sub(r"[ \t]+", " ", "\n".join(merged)).strip(), pasal, bab

### Metadata enrichment

Setiap halaman diperkaya metadata yang dipakai untuk **filtering** dan **sitasi**:

| Field | Contoh | Kegunaan |
|---|---|---|
| `peraturan_id`, `jenis`, `nomor`, `tahun` | `PP-35-2021`, `PP`, `35`, `2021` | metadata filtering |
| `singkatan`, `judul`, `topik` | `PP 35/2021`, `...Waktu Kerja...`, `ketenagakerjaan` | sitasi & konteks LLM |
| `page`, `total_pages`, `source` | `19`, `56`, `PP Nomor 35 Tahun 2021.pdf` | sitasi halaman |
| `pasal`, `bab` | `31`, `IV` | pasal yang berlaku di halaman itu (termasuk lanjutan dari halaman sebelumnya) |

In [6]:
pages = []
load_stats = []
for file_name, doc_meta in DOC_CATALOG.items():
    raw_pages = PyMuPDFLoader(os.path.join(DOC_DIR, file_name)).load()
    current_pasal, current_bab, kept = "-", "-", 0
    for raw in raw_pages:
        text, pasal_started, bab_started = clean_page(raw.page_content)
        # pasal dari halaman sebelumnya masih berlaku bila halaman tidak diawali "Pasal N"
        carried = [current_pasal] if current_pasal != "-" and not text.startswith("Pasal") else []
        pasal_on_page = list(dict.fromkeys(carried + pasal_started))
        current_pasal = pasal_started[-1] if pasal_started else current_pasal
        current_bab = bab_started[-1] if bab_started else current_bab
        if len(text) < 100:
            continue
        kept += 1
        pages.append(Document(page_content=text, metadata={
            **doc_meta,
            "source": file_name,
            "page": raw.metadata["page"] + 1,
            "total_pages": raw.metadata["total_pages"],
            "pasal": ", ".join(pasal_on_page) or "-",
            "bab": current_bab,
        }))
    load_stats.append({"dokumen": doc_meta["singkatan"], "halaman PDF": len(raw_pages), "halaman berisi teks": kept})

display(pd.DataFrame(load_stats).set_index("dokumen"))
print("Total halaman dimuat:", len(pages))

,halaman PDF,halaman berisi teks
dokumen,,
UU 6/2023,1127,1127
PP 35/2021,56,56
PP 51/2023,27,27
PP 5/2021,739,349


Total halaman dimuat: 1559


In [7]:
sample = next(p for p in pages if p.metadata["peraturan_id"] == "PP-35-2021" and p.metadata["page"] == 17)
print(json.dumps(sample.metadata, indent=2, ensure_ascii=False))
print(sample.page_content[:700])

{
  "peraturan_id": "PP-35-2021",
  "jenis": "PP",
  "nomor": 35,
  "tahun": 2021,
  "singkatan": "PP 35/2021",
  "judul": "Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja",
  "topik": "ketenagakerjaan",
  "source": "PP Nomor 35 Tahun 2021.pdf",
  "page": 17,
  "total_pages": 56,
  "pasal": "25, 26, 27, 28",
  "bab": "IV"
}
Bagian Ketiga Waktu Kerja Lembur
Pasal 26
(1) Waktu Kerja Lembur hanya dapat dilakukan paling lama 4 (empat)jam dalam 1 (satu) hari dan 18 (delapan belas) jam dalam 1 (satu) minggu.
(2) Ketentuan Waktu Kerja Lembur sebagaimana dimaksud pada ayat (1) tidak termasuk kerja lembur yang dilakukan pada waktu istirahat mingguan dan/atau hari libur resmi.
Pasal 27
(1) Pengusaha yang mempekerjakan Pekerja/Buruh melebihi waktu kerja sebagaimana dimaksud dalam Pasal 2l ayat (2), wajib membayar Upah Kerja Lembur.
(2) Kewajiban membayar Upah Kerja Lembur dikecualikan bagi Pekerja/Buruh dalam golongan jabatan tertentu.
(3)

## 2. Text Splitting: Parent Chunk & Child Chunk

Ukuran chunk dan overlap ditentukan eksplisit:

| Splitter | chunk_size | chunk_overlap | Peran |
|---|---|---|---|
| `parent_splitter` | 2000 | 200 | konteks untuk LLM; mayoritas halaman UU/PP < 2000 karakter sehingga 1 parent ≈ 1 halaman utuh |
| `child_splitter` | 400 | 80 | unit pencarian vektor (embedding lebih presisi pada potongan kecil) |

In [8]:
PARENT_CHUNK_SIZE, PARENT_CHUNK_OVERLAP = 2000, 200
CHILD_CHUNK_SIZE, CHILD_CHUNK_OVERLAP = 400, 80

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=PARENT_CHUNK_SIZE, chunk_overlap=PARENT_CHUNK_OVERLAP, separators=["\nPasal ", "\n", ". ", " ", ""]
)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHILD_CHUNK_SIZE, chunk_overlap=CHILD_CHUNK_OVERLAP, separators=["\n", ". ", " ", ""]
)

parent_docs = parent_splitter.split_documents(pages)
for i, doc in enumerate(parent_docs):
    doc.metadata["parent_id"] = f"{doc.metadata['peraturan_id']}-hal{doc.metadata['page']}-{i}"
    doc.metadata["char_len"] = len(doc.page_content)

n_children = sum(len(child_splitter.split_text(d.page_content)) for d in parent_docs)
parent_lengths = pd.Series([d.metadata["char_len"] for d in parent_docs])
print(f"Parent chunks: {len(parent_docs)} | Child chunks: {n_children}")
print(f"Panjang parent (karakter): median {parent_lengths.median():.0f}, maks {parent_lengths.max()}")
display(pd.Series(Counter(d.metadata["singkatan"] for d in parent_docs), name="parent chunks").to_frame())

Parent chunks: 1573 | Child chunks: 7008
Panjang parent (karakter): median 1260, maks 1987


,parent chunks
UU 6/2023,1140
PP 35/2021,56
PP 51/2023,28
PP 5/2021,349


## 3. Embedding Open-Source + Vector DB Lokal (ChromaDB) + Parent-Child Retriever

- Embedding: `BAAI/bge-m3` (open-source, multilingual, mendukung Bahasa Indonesia).
- Vector DB: **ChromaDB** lokal (in-process) menyimpan vektor **child chunks** (cosine distance).
- `ParentDocumentRetriever`: mencari child chunk yang paling mirip, lalu mengembalikan **parent chunk**
  dari `InMemoryStore` sebagai konteks.

```text
parent chunk (≈1 halaman, InMemoryStore)          <- dikirim ke LLM
    └── child chunk 400 karakter (ChromaDB)       <- dicocokkan dengan query
```

In [9]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
)

COLLECTION = "uu_pp_child_chunks"
chroma_client = chromadb.EphemeralClient()
try:
    chroma_client.delete_collection(COLLECTION)        # aman di-run ulang tanpa duplikasi vektor
except Exception:
    pass
vectorstore = Chroma(
    client=chroma_client,
    collection_name=COLLECTION,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"},
)
docstore = InMemoryStore()

parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,   # parent sudah dipotong di atas -> tidak perlu parent_splitter lagi
    search_kwargs={"k": 20},         # 20 child chunk teratas -> beberapa parent unik
)

start = time.time()
parent_ids = [d.metadata["parent_id"] for d in parent_docs]
BATCH = 256
for i in range(0, len(parent_docs), BATCH):
    parent_retriever.add_documents(parent_docs[i:i + BATCH], ids=parent_ids[i:i + BATCH])
print(f"Child vectors di ChromaDB: {vectorstore._collection.count()} | waktu: {time.time() - start:.0f} detik")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Child vectors di ChromaDB: 7008 | waktu: 107 detik


## 4. Uji Retrieval

In [10]:
def show_docs(docs, scores=None, max_chars=220):
    rows = []
    for i, doc in enumerate(docs):
        m = doc.metadata
        rows.append({
            "#": i + 1,
            "dokumen": m.get("singkatan", m.get("title", "web")),
            "pasal": m.get("pasal", "-"),
            "halaman": m.get("page", "-"),
            **({"skor": round(scores[i], 4)} if scores is not None else {}),
            "cuplikan": doc.page_content[:max_chars].replace("\n", " ") + " ...",
        })
    display(pd.DataFrame(rows).set_index("#"))


TEST_QUERY = "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"

print("Child chunks (vector search + cosine distance):")
child_hits = vectorstore.similarity_search_with_score(TEST_QUERY, k=5)
show_docs([d for d, _ in child_hits], [s for _, s in child_hits])

print("Parent chunks dari ParentDocumentRetriever:")
show_docs(parent_retriever.invoke(TEST_QUERY)[:5])

Child chunks (vector search + cosine distance):


,dokumen,pasal,halaman,skor,cuplikan
#,,,,,
1,PP 35/2021,31,19,0.3239,"(3) Perusahaan yang mempekerjakan Pekerja/Buruh sebagaimana dimaksud pada ayat (1) wajib membayar Upah Kerja Lembur, apabila kerja lembur dilakukan pada hari istirahat mingguan dan/atau hari libur resmi untuk waktu kerja ..."
2,PP 35/2021,"28, 30, 31",18,0.3244,(1) Perusahaan yang mempekerjakan Pekerja/Buruh selama Waktu Kerja Lembur berkewajiban: a. membayar Upah Kerja Lembur; b. memberi kesempatan untuk istirahat secukupnya; dan c. memberikan makanan dan minuman paling sediki ...
3,PP 35/2021,31,19,0.3412,"b. untuk setiap ja- kerja lembur berikutnya, sebesar 2 (dua) kali Upah sejam. (2) Perusahaan yang mempekerjakan Pekerja/Buruh sebagaimana dimaksud pada ayat (1) wajib membayar Upah Kerja Lembur, apabila kerja lembur dila ..."
4,PP 35/2021,31,19,0.3480,"a. perhitungan Upah Kerja Lembur dilaksanakan sebagai berikut: 1. jam pertama sampai dengan jam ketujuh, dibayar 2 (dua) kali Upah sejam; 2. jam kedelapan, dibayar 3 (tiga) kali Upah sejam; dan 3. jam kesembilan, jam kes ..."
5,PP 35/2021,"31, 32, 33, 34",20,0.3495,"b. jam kesembilan, dibayar 3 (tiga) kali Upah sejam; dan c. jam kesepuluh, jam kesebelas, dan jam kedua belas, dibayar 4 (empat) kali Upah sejam. Pasal 32 (1) Perhitungan Upah Kerja Lembur didasarkan pada Upah bulanan. ( ..."


Parent chunks dari ParentDocumentRetriever:


,dokumen,pasal,halaman,cuplikan
#,,,,
1,PP 35/2021,31,19,"a. untuk ja- kerja lembur pertama sebesar 1,5 (satu koma lima) kali Upah sejam; dan b. untuk setiap ja- kerja lembur berikutnya, sebesar 2 (dua) kali Upah sejam. (2) Perusahaan yang mempekerjakan Pekerja/Buruh sebagaiman ..."
2,PP 35/2021,"28, 30, 31",18,(2) Perintah dan persetujuan sebagaimana dimaksud pada ayat (1) dapat dibuat dalam bentuk daftar Pekerja/Buruh yang bersedia bekerja lembur yang ditandatangani oleh Pekerja/Buruh yang bersangkutan dan Pengusaha. (3) Peng ...
3,PP 35/2021,"31, 32, 33, 34",20,"b. jam kesembilan, dibayar 3 (tiga) kali Upah sejam; dan c. jam kesepuluh, jam kesebelas, dan jam kedua belas, dibayar 4 (empat) kali Upah sejam. Pasal 32 (1) Perhitungan Upah Kerja Lembur didasarkan pada Upah bulanan. ( ..."
4,PP 35/2021,"25, 26, 27, 28",17,Bagian Ketiga Waktu Kerja Lembur Pasal 26 (1) Waktu Kerja Lembur hanya dapat dilakukan paling lama 4 (empat)jam dalam 1 (satu) hari dan 18 (delapan belas) jam dalam 1 (satu) minggu. (2) Ketentuan Waktu Kerja Lembur sebag ...
5,PP 35/2021,1,3,"b. usaha-usaha sosial dan usaha-usaha lain yang mempunyai pengurus dan mempekerjakan orang lain dengan membayar Upah atau imbalan dalam bentuk lain. Serikat Pekerja/Serikat Buruh adalah organisasi yang dibentuk dari, ole ..."


## 5. Metadata Filtering

Filter diterapkan langsung di ChromaDB (pre-filter) sebelum pencarian vektor. `detect_metadata_filter`
membaca rujukan eksplisit di pertanyaan (mis. "PP 35/2021", "UU No. 6 Tahun 2023") sehingga pencarian
otomatis dibatasi ke dokumen itu.

In [11]:
def to_chroma_filter(metadata_filter):
    """{"jenis": "PP", "tahun": 2021} -> sintaks filter ChromaDB."""
    if not metadata_filter:
        return None
    clauses = [{key: value} for key, value in metadata_filter.items()]
    return clauses[0] if len(clauses) == 1 else {"$and": clauses}


def matches_filter(metadata, metadata_filter):
    return not metadata_filter or all(metadata.get(k) == v for k, v in metadata_filter.items())


REFERENCE_RE = re.compile(
    r"\b(PP|UU|Peraturan Pemerintah|Undang[- ]Undang)\s*(?:No(?:mor|\.)?\s*)?(\d+)\s*(?:/|Tahun)\s*(\d{4})",
    re.IGNORECASE,
)


def detect_metadata_filter(question):
    """Rujukan peraturan eksplisit di pertanyaan -> filter peraturan_id (hanya untuk 4 dokumen yang tersedia)."""
    known = {m["peraturan_id"] for m in DOC_CATALOG.values()}
    for jenis, nomor, tahun in REFERENCE_RE.findall(question):
        jenis = "PP" if jenis.upper().startswith("P") else "UU"
        peraturan_id = f"{jenis}-{int(nomor)}-{tahun}"
        if peraturan_id in known:
            return {"peraturan_id": peraturan_id}
    return None


for question in [
    "Berdasarkan PP 51/2023, bagaimana formula penghitungan upah minimum?",
    "Menurut Peraturan Pemerintah Nomor 35 Tahun 2021, berapa lama maksimal PKWT?",
    TEST_QUERY,
]:
    print(f"{question!r:90.90} -> {detect_metadata_filter(question)}")

'Berdasarkan PP 51/2023, bagaimana formula penghitungan upah minimum?'                     -> {'peraturan_id': 'PP-51-2023'}
'Menurut Peraturan Pemerintah Nomor 35 Tahun 2021, berapa lama maksimal PKWT?'             -> {'peraturan_id': 'PP-35-2021'}
'Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uan -> None


In [12]:
question = "Bagaimana formula penghitungan upah minimum?"
for metadata_filter in [None, {"peraturan_id": "PP-51-2023"}, {"jenis": "UU"}]:
    hits = vectorstore.similarity_search(question, k=3, filter=to_chroma_filter(metadata_filter))
    print(f"filter = {metadata_filter}")
    show_docs(hits, max_chars=150)

filter = None


,dokumen,pasal,halaman,cuplikan
#,,,,
1,UU 6/2023,"88C, 88D, 88E, 88F",563,"(2) Formula penghitungan Upah minimum sebagaimana dimaksud pada ayat (1) mempertimbangkan variabel pertumbuhan ekonomi, inflasi, dan indeks tertentu. ..."
2,UU 6/2023,"88C, 88D, 88E, 88F",563,(7) Ketentuan lebih lanjut mengenai tata cara penetapan Upah minimum sebagaimana dimaksud pada ayat (4) dan syarat tertentu sebagaimana dimaksud pada ...
3,PP 51/2023,81C,17,". Undang-Undang Nomor 6 Tahun 2023 mengatur bahwa formula penghitungan Upah minimum mempertimbangkan variabel pertumbuhan ekonomi, inflasi, dan indeks ..."


filter = {'peraturan_id': 'PP-51-2023'}


,dokumen,pasal,halaman,cuplikan
#,,,,
1,PP 51/2023,81C,17,". Undang-Undang Nomor 6 Tahun 2023 mengatur bahwa formula penghitungan Upah minimum mempertimbangkan variabel pertumbuhan ekonomi, inflasi, dan indeks ..."
2,PP 51/2023,"31B, 32",9,Pasal 31B (1) Penghitungan Upah minimum bagi kabupaten/kota yang belum memiliki Upah minimum kabupaten/kota sebagaimana dimaksud dalam Pasal 30 ayat ( ...
3,PP 51/2023,25,3,(3) Indeks tertentu sebagaimana dimaksud pada ayat (2) yang disimbolkan dengan o merupakan variabel yang mewakili kontribusi tenaga kerja terhadap per ...


filter = {'jenis': 'UU'}


,dokumen,pasal,halaman,cuplikan
#,,,,
1,UU 6/2023,"88C, 88D, 88E, 88F",563,"(2) Formula penghitungan Upah minimum sebagaimana dimaksud pada ayat (1) mempertimbangkan variabel pertumbuhan ekonomi, inflasi, dan indeks tertentu. ..."
2,UU 6/2023,"88C, 88D, 88E, 88F",563,(7) Ketentuan lebih lanjut mengenai tata cara penetapan Upah minimum sebagaimana dimaksud pada ayat (4) dan syarat tertentu sebagaimana dimaksud pada ...
3,UU 6/2023,"88C, 88D, 88E, 88F",563,(2) Pengusaha dilarang membayar Upah lebih rendah dari Upah minimum. Pasal 88F Dalam keadaan tertentu Pemerintah dapat menetapkan formula penghitungan ...


## 6. Ensemble Retriever: BM25 (keyword) + Parent-Child (semantik)

- **BM25** menangkap istilah hukum persis (mis. "Upah Kerja Lembur", "PKWT", nomor pasal).
- **Retriever semantik** menangkap parafrase bahasa sehari-hari ("beresin laporan", "uang lembur").
- Bobot: `[0.4, 0.6]` (BM25, semantik). Semantik diberi bobot lebih besar karena pertanyaan Legal Team
  cenderung bahasa percakapan, sedangkan BM25 tetap menjaga istilah teknis. Keduanya mengembalikan
  **parent chunk** yang sama, sehingga hasil digabung dan dideduplikasi lewat `id_key="parent_id"`.

In [13]:
STOPWORDS_ID = set("""yang dan di ke dari dalam untuk pada dengan atau ini itu adalah sebagaimana dimaksud ayat
huruf oleh tidak dapat akan bagi sebagai juga serta tersebut telah atas secara para saya apakah kemarin anda kami
bisa mau dapat""".split())


def tokenize_id(text):
    return [t for t in re.findall(r"[a-z0-9]+", text.lower()) if t not in STOPWORDS_ID and len(t) > 1]


BM25_K = 10
ENSEMBLE_WEIGHTS = [0.4, 0.6]   # [BM25, semantik]

bm25_retriever = BM25Retriever.from_documents(parent_docs, k=BM25_K, preprocess_func=tokenize_id)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, parent_retriever],
    weights=ENSEMBLE_WEIGHTS,
    id_key="parent_id",
)

ensemble_docs = ensemble_retriever.invoke(TEST_QUERY)
print(f"Ensemble mengembalikan {len(ensemble_docs)} dokumen (>= 5)")
show_docs(ensemble_docs[:8])

Ensemble mengembalikan 10 dokumen (>= 5)


,dokumen,pasal,halaman,cuplikan
#,,,,
1,PP 35/2021,31,19,"a. untuk ja- kerja lembur pertama sebesar 1,5 (satu koma lima) kali Upah sejam; dan b. untuk setiap ja- kerja lembur berikutnya, sebesar 2 (dua) kali Upah sejam. (2) Perusahaan yang mempekerjakan Pekerja/Buruh sebagaiman ..."
2,PP 35/2021,"28, 30, 31",18,(2) Perintah dan persetujuan sebagaimana dimaksud pada ayat (1) dapat dibuat dalam bentuk daftar Pekerja/Buruh yang bersedia bekerja lembur yang ditandatangani oleh Pekerja/Buruh yang bersangkutan dan Pengusaha. (3) Peng ...
3,PP 35/2021,"25, 26, 27, 28",17,Bagian Ketiga Waktu Kerja Lembur Pasal 26 (1) Waktu Kerja Lembur hanya dapat dilakukan paling lama 4 (empat)jam dalam 1 (satu) hari dan 18 (delapan belas) jam dalam 1 (satu) minggu. (2) Ketentuan Waktu Kerja Lembur sebag ...
4,PP 35/2021,"31, 32, 33, 34",20,"b. jam kesembilan, dibayar 3 (tiga) kali Upah sejam; dan c. jam kesepuluh, jam kesebelas, dan jam kedua belas, dibayar 4 (empat) kali Upah sejam. Pasal 32 (1) Perhitungan Upah Kerja Lembur didasarkan pada Upah bulanan. ( ..."
5,PP 35/2021,1,3,"b. usaha-usaha sosial dan usaha-usaha lain yang mempunyai pengurus dan mempekerjakan orang lain dengan membayar Upah atau imbalan dalam bentuk lain. Serikat Pekerja/Serikat Buruh adalah organisasi yang dibentuk dari, ole ..."
6,UU 6/2023,"78, 79",559,(2) Pengusaha yang mempekerjakan Pekerja/Buruh melebihi waktu kerja sebagaimana dimaksud pada ayat (1) wajib membayar Upah kerja lembur. (3) Ketentuan waktu kerja lembur sebagaimana dimaksud pada ayat (1) hurrrf b tidak ...
7,PP 35/2021,"34, 35, 36",21,Pasal 34 (1) Dalam hal Perusahaan telah melaksanakan pembayaran Upah Kerja Lembur dengan sebutan lain dan nilai perhitungan Upah Kerja Lembur sama dengan atau lebih baik maka perhitungan Upah Kerja Lembur tetap berlaku. ...
8,PP 35/2021,"66, 1, 2, 3, 4, 5",44,d. pelindungan Pekerja/Buruh dan perizinan berusaha pada kegiatan alih daya; e. waktu kerja pada sektor usaha atau pekerjaan tertentu; f. Waktu Kerja Lembur dan Upah Kerja Lembur; g. batasan Perusahaan tertentu yang dapa ...


## 7. Memuat Model Hasil Fine-tuning (Kriteria 1)

Model yang dimuat adalah **model Kriteria 1**: hasil SFT (`SFT_REPO`) yang dilanjutkan dengan GRPO
(`GRPO_REPO`). Keduanya hasil training sendiri dan di-*push* ke Hugging Face, bukan model baru.
Ganti `LLM_REPO = SFT_REPO` untuk memakai model SFT murni.

In [14]:
from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient


def get_hf_token():
    """Token Hugging Face (akses write) dari Colab Secrets / Kaggle Secrets / env var / input manual."""
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    try:
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        return hf_token
    except Exception:
        pass
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    from getpass import getpass
    return getpass("Masukkan Hugging Face token (write): ")


HF_TOKEN = get_hf_token()
login(token=HF_TOKEN)
HF_USERNAME = whoami(token=HF_TOKEN)["name"]
print("Login Hugging Face sebagai:", HF_USERNAME)

Login Hugging Face sebagai: Fepriyadi


In [15]:
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"   # arsitektur Qwen2 (didukung Unsloth), task: Text Generation
SFT_REPO = f"{HF_USERNAME}/Qwen2.5-3B-Instruct-Alpaca-ID-SFT"     # hasil Kriteria 1 (SFT)
GRPO_REPO = f"{HF_USERNAME}/Qwen2.5-3B-Instruct-Alpaca-ID-GRPO"   # hasil Kriteria 1 Advanced (GRPO)
SEED = 3407

THINK_SYSTEM_PROMPT = (
    "Anda adalah asisten AI berbahasa Indonesia. Sebelum menjawab, tuliskan proses berpikir Anda "
    "langkah demi langkah di dalam tag <think> ... </think>. Setelah tag </think>, tuliskan jawaban akhir "
    "yang jelas dan ringkas dalam Bahasa Indonesia.\n"
    "Format:\n<think>\nproses berpikir\n</think>\njawaban akhir"
)

LLM_REPO = GRPO_REPO
print("Generator:", LLM_REPO)

Generator: Fepriyadi/Qwen2.5-3B-Instruct-Alpaca-ID-GRPO


In [16]:
from transformers import BitsAndBytesConfig

llm, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LLM_REPO,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    ),
    use_exact_model_name=True,
    token=HF_TOKEN,
)
FastLanguageModel.for_inference(llm)
print(f"VRAM terpakai: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

==((====))==  Unsloth 2026.10.8: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

VRAM terpakai: 4.04 GB


In [17]:
@torch.inference_mode()
def generate_chat(messages, max_new_tokens=512, temperature=0.2, num_return_sequences=1, assistant_prefix=""):
    """Generate dari model fine-tuned. `assistant_prefix` ditempel setelah header assistant (prefill)."""
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + assistant_prefix
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(llm.device)
    outputs = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 1e-5),
        top_p=0.9,
        # 1.0 = tanpa penalti. Penalti > 1 juga berlaku pada token prompt, sehingga menyalin kalimat pasal dari
        # konteks justru dihukum dan model memilih parafrase (sumber salah baca "paling lama 4 jam").
        repetition_penalty=1.0,
        num_return_sequences=num_return_sequences,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    prompt_len = inputs["input_ids"].shape[1]
    return [assistant_prefix + tokenizer.decode(o[prompt_len:], skip_special_tokens=True) for o in outputs]

In [18]:
THINK_OPEN, THINK_CLOSE = "<think>", "</think>"


def completion_text(completion):
    """GRPOTrainer mengirim completion sebagai list pesan (format percakapan) atau string biasa."""
    if isinstance(completion, list):
        return completion[0]["content"] if completion else ""
    return completion


def extract_thinking(text):
    """Isi <think>...</think>. Toleran terhadap penalaran yang terpotong max_completion_length (tanpa </think>)."""
    if THINK_OPEN not in text:
        return ""
    after_open = text.split(THINK_OPEN, 1)[1]
    return after_open.split(THINK_CLOSE, 1)[0].strip()


def extract_final_answer(text):
    """Jawaban akhir = teks setelah </think> terakhir.
    Ada <think> tapi tidak ditutup -> model belum selesai berpikir -> belum ada jawaban akhir.
    Tidak ada tag sama sekali -> seluruh teks dianggap jawaban."""
    if THINK_CLOSE in text:
        return text.rsplit(THINK_CLOSE, 1)[1].strip()
    if THINK_OPEN in text:
        return ""
    return text.strip()

### Prompt RAG dengan `{context}` dan `{question}`

System prompt memakai format `<think>` yang sama persis dengan saat GRPO. Generasi diawali prefill `<think>\n`
(teknik yang sama dengan template DeepSeek-R1) agar blok penalaran selalu dibuka; isi penalaran,
penutup `</think>`, dan jawaban akhir tetap ditulis model.

**Kenapa prompt meminta kutipan dulu.** Pada run sebelumnya konteks sudah memuat jawaban test case lembur
(PP 35/2021 Pasal 26 ayat (1) "paling lama 4 (empat) jam" dan Pasal 27 ayat (1) "wajib membayar Upah Kerja
Lembur"), tetapi model memparafrasekan dari ingatan dan membalik syaratnya:

```text
paling lama 4 jam  = maksimal 4 jam  -> lembur 3 jam sah dan wajib dibayar
4 jam atau lebih   = minimal 4 jam   -> "3 jam tidak berhak"   (yang ditulis model)
```

Jawaban akhirnya juga bertentangan dengan `<think>` ("tidak dapat menentukan" lalu "tidak berhak").
Karena itu prompt sekarang mewajibkan tiga langkah di dalam `<think>`: kutip kalimat pasal beserta `[n]`,
cocokkan fakta pertanyaan dengan syarat di kutipan, baru simpulkan. Jawaban akhir harus sama dengan
kesimpulan itu, dan "tidak ditemukan" hanya boleh dipakai bila tidak ada kutipan yang relevan.
`repetition_penalty` di `generate_chat` juga diturunkan ke 1.0 agar menyalin kalimat pasal tidak dihukum.

In [19]:
RAG_SYSTEM_PROMPT = THINK_SYSTEM_PROMPT + (
    "\nAnda adalah asisten hukum untuk Legal Team perusahaan di Indonesia. Jawab berdasarkan konteks peraturan "
    "yang diberikan dan cantumkan nomor sumber seperti [1] atau [2] di jawaban akhir."
)

RAG_PROMPT = PromptTemplate.from_template(
    """Gunakan HANYA konteks berikut untuk menjawab pertanyaan.
Setiap potongan konteks diberi nomor sumber [n].

Di dalam <think>, kerjakan tiga langkah ini:
1. Kutip kata demi kata kalimat pasal yang relevan beserta nomor sumbernya, misalnya: [2] Pasal 26 ayat (1): "...".
2. Cocokkan fakta pada pertanyaan (angka, jabatan, waktu) dengan syarat di kutipan. Perhatikan kata batas:
   "paling lama" berarti maksimal, "paling sedikit" berarti minimal.
3. Tulis kesimpulan.

Setelah </think>, tulis jawaban akhir yang sama dengan kesimpulan di <think>, lalu sebutkan nama peraturan,
pasal, dan nomor sumber [n] yang Anda pakai.
Hanya jika langkah 1 tidak menemukan kutipan yang relevan, katakan bahwa informasi tersebut tidak ditemukan di dokumen.

Konteks:
{context}

Pertanyaan: {question}"""
)
THINK_PREFIX = "<think>\n"


def format_context(docs):
    blocks = []
    for i, doc in enumerate(docs, start=1):
        m = doc.metadata
        if m.get("source_type") == "web":
            header = f"[{i}] {m['title']} ({m['url']})"
        else:
            header = f"[{i}] {m['singkatan']} tentang {m['judul']} — Pasal {m['pasal']}, halaman {m['page']}"
        blocks.append(f"{header}\n{doc.page_content}")
    return "\n\n".join(blocks)


print(RAG_PROMPT.format(context=format_context(ensemble_docs[:1]), question=TEST_QUERY)[:900])

Gunakan HANYA konteks berikut untuk menjawab pertanyaan.
Setiap potongan konteks diberi nomor sumber [n].

Di dalam <think>, kerjakan tiga langkah ini:
1. Kutip kata demi kata kalimat pasal yang relevan beserta nomor sumbernya, misalnya: [2] Pasal 26 ayat (1): "...".
2. Cocokkan fakta pada pertanyaan (angka, jabatan, waktu) dengan syarat di kutipan. Perhatikan kata batas:
   "paling lama" berarti maksimal, "paling sedikit" berarti minimal.
3. Tulis kesimpulan.

Setelah </think>, tulis jawaban akhir yang sama dengan kesimpulan di <think>, lalu sebutkan nama peraturan,
pasal, dan nomor sumber [n] yang Anda pakai.
Hanya jika langkah 1 tidak menemukan kutipan yang relevan, katakan bahwa informasi tersebut tidak ditemukan di dokumen.

Konteks:
[1] PP 35/2021 tentang Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja — Pasal 31, halaman 19


### Generasi RAG dasar (Ensemble top-5 → LLM)

In [20]:
basic_context = format_context(ensemble_docs[:5])
basic_messages = [
    {"role": "system", "content": RAG_SYSTEM_PROMPT},
    {"role": "user", "content": RAG_PROMPT.format(context=basic_context, question=TEST_QUERY)},
]
basic_output = generate_chat(basic_messages, assistant_prefix=THINK_PREFIX)[0]
print(basic_output)

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>
Pasal 26 ayat (1) menyatakan bahwa waktu kerja lembur hanya dapat dilakukan paling lama 4 jam dalam 1 hari dan 18 jam dalam 1 minggu. Pasal 31 ayat (1) menyatakan bahwa perusahaan yang mempekerjakan Pekerja/Buruh melebihi waktu kerja sebagaimana dimaksud dalam Pasal 2l ayat (2) wajib membayar Upah Kerja Lembur. Pasal 31 ayat (1) menyatakan bahwa Upah Kerja Lembur hanya dapat dilakukan paling lama 4 jam dalam 1 hari dan 18 jam dalam 1 minggu. Dalam kasus ini, waktu kerja lembur sebesar 3 jam, sehingga berdasarkan Pasal 26 ayat (1) dan Pasal 31 ayat (1), Anda berhak mendapatkan Upah Kerja Lembur.
</think>
[1] PP 35/2021 tentang Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja — Pasal 26, halaman 19.


## 8. HyDE (Hypothetical Document Embeddings)

Pertanyaan percakapan ("beresin laporan", "uang lembur") jauh dari gaya bahasa pasal UU.
HyDE meminta LLM menulis **2 jawaban hipotetis** bergaya pasal peraturan. Embedding pertanyaan dan kedua
jawaban hipotetis dirata-rata, lalu vektor gabungan itu dipakai untuk mencari child chunk di ChromaDB.

Jawaban hipotetis boleh salah fakta; yang dimanfaatkan hanya kedekatan kosakatanya dengan teks pasal.

In [21]:
HYDE_SYSTEM_PROMPT = (
    "Anda adalah ahli hukum ketenagakerjaan Indonesia. Tulis satu paragraf singkat (3-5 kalimat) bergaya pasal "
    "peraturan perundang-undangan Indonesia yang kemungkinan besar menjawab pertanyaan pengguna. "
    "Langsung tulis paragrafnya."
)
N_HYPOTHESES = 2


def generate_hypotheses(question, n=N_HYPOTHESES):
    messages = [{"role": "system", "content": HYDE_SYSTEM_PROMPT}, {"role": "user", "content": question}]
    outputs = generate_chat(messages, max_new_tokens=220, temperature=0.7, num_return_sequences=n)
    # model GRPO terbiasa menulis <think>; ambil jawaban akhir, atau isi penalarannya jika belum selesai
    return [extract_final_answer(o) or extract_thinking(o) or o for o in outputs]

In [22]:
class HyDEParentRetriever(BaseRetriever):
    """Vector search dengan embedding gabungan (query + jawaban hipotetis) -> parent chunks."""

    vectorstore: Any
    docstore: Any
    embeddings: Any
    hypothesis_fn: Any
    id_key: str = "doc_id"
    k_child: int = 20
    metadata_filter: Optional[dict] = None
    last_hypotheses: list = []

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun):
        hypotheses = self.hypothesis_fn(query)
        self.last_hypotheses = hypotheses
        vectors = [self.embeddings.embed_query(query)] + self.embeddings.embed_documents(hypotheses)
        combined = torch.tensor(vectors).mean(dim=0)
        combined = torch.nn.functional.normalize(combined, dim=0).tolist()
        children = self.vectorstore.similarity_search_by_vector(
            combined, k=self.k_child, filter=to_chroma_filter(self.metadata_filter)
        )
        ordered_ids = list(dict.fromkeys(child.metadata[self.id_key] for child in children))
        return [doc for doc in self.docstore.mget(ordered_ids) if doc is not None]


class FilteredBM25Retriever(BaseRetriever):
    """BM25 atas parent chunks yang lolos metadata filter (indeks per filter di-cache)."""

    documents: list
    k: int = BM25_K
    metadata_filter: Optional[dict] = None
    cache: Any = None

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun):
        key = json.dumps(self.metadata_filter, sort_keys=True)
        if key not in self.cache:
            docs = [d for d in self.documents if matches_filter(d.metadata, self.metadata_filter)]
            self.cache[key] = BM25Retriever.from_documents(docs, k=self.k, preprocess_func=tokenize_id)
        return self.cache[key].invoke(query)


BM25_CACHE = {}


def build_hybrid_retriever(metadata_filter=None, use_hyde=True):
    """Ensemble BM25 + semantik (HyDE atau query biasa), keduanya menghormati metadata filter."""
    sparse = FilteredBM25Retriever(documents=parent_docs, metadata_filter=metadata_filter, cache=BM25_CACHE)
    if use_hyde:
        dense = HyDEParentRetriever(
            vectorstore=vectorstore, docstore=docstore, embeddings=embedding_model,
            hypothesis_fn=generate_hypotheses, id_key=parent_retriever.id_key, metadata_filter=metadata_filter,
        )
    else:
        dense = ParentDocumentRetriever(
            vectorstore=vectorstore, docstore=docstore, child_splitter=child_splitter,
            search_kwargs={"k": 20, "filter": to_chroma_filter(metadata_filter)},
        )
    return EnsembleRetriever(retrievers=[sparse, dense], weights=ENSEMBLE_WEIGHTS, id_key="parent_id"), dense

In [23]:
hyde_ensemble, hyde_dense = build_hybrid_retriever(use_hyde=True)
hyde_docs = hyde_ensemble.invoke(TEST_QUERY)

for i, hypothesis in enumerate(hyde_dense.last_hypotheses, start=1):
    print(f"--- Jawaban hipotetis {i} ---\n{hypothesis}\n")
print(f"Ensemble (BM25 + HyDE) mengembalikan {len(hyde_docs)} dokumen")
show_docs(hyde_docs[:8])

Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Jawaban hipotetis 1 ---
Menurut Pasal 288 Undang-Undang Nomor 13 tahun 2013 tentang Ketenagakerjaan, seorang pekerja berhak mendapatkan uang lembur jika bekerja di luar jam kerja yang ditentukan dalam surat pernyataan kerja. Jika pekerja telah menandatangani surat pernyataan kerja yang mencakup jam kerja yang ditentukan, pekerja harus mendapatkan uang lembur jika bekerja di luar jam kerja tersebut. Namun, jika pekerja tidak menandatangani surat pernyataan kerja atau jika jam kerja tidak ditentukan, pekerja tidak berhak mendapatkan uang lembur. Dalam kasus Anda, jika Anda telah menandatangani surat pernyataan kerja yang mencakup jam kerja yang ditentukan, Anda berhak mendapatkan uang lembur. Namun, jika jam kerja tidak dit

--- Jawaban hipotetis 2 ---
Menurut Pasal 222 KUHP, seorang pekerja yang bekerja lebih dari jam kerja yang ditentukan dalam kontrak kerja atau kontrak kerja yang tidak ditentukan harus mendapatkan upah tambahan. Jika Anda bekerja lebih dari jam kerja yang ditentu

,dokumen,pasal,halaman,cuplikan
#,,,,
1,PP 35/2021,31,19,"a. untuk ja- kerja lembur pertama sebesar 1,5 (satu koma lima) kali Upah sejam; dan b. untuk setiap ja- kerja lembur berikutnya, sebesar 2 (dua) kali Upah sejam. (2) Perusahaan yang mempekerjakan Pekerja/Buruh sebagaiman ..."
2,PP 35/2021,"25, 26, 27, 28",17,Bagian Ketiga Waktu Kerja Lembur Pasal 26 (1) Waktu Kerja Lembur hanya dapat dilakukan paling lama 4 (empat)jam dalam 1 (satu) hari dan 18 (delapan belas) jam dalam 1 (satu) minggu. (2) Ketentuan Waktu Kerja Lembur sebag ...
3,UU 6/2023,"78, 79",559,(2) Pengusaha yang mempekerjakan Pekerja/Buruh melebihi waktu kerja sebagaimana dimaksud pada ayat (1) wajib membayar Upah kerja lembur. (3) Ketentuan waktu kerja lembur sebagaimana dimaksud pada ayat (1) hurrrf b tidak ...
4,PP 35/2021,"28, 30, 31",18,(2) Perintah dan persetujuan sebagaimana dimaksud pada ayat (1) dapat dibuat dalam bentuk daftar Pekerja/Buruh yang bersedia bekerja lembur yang ditandatangani oleh Pekerja/Buruh yang bersangkutan dan Pengusaha. (3) Peng ...
5,PP 35/2021,1,3,"b. usaha-usaha sosial dan usaha-usaha lain yang mempunyai pengurus dan mempekerjakan orang lain dengan membayar Upah atau imbalan dalam bentuk lain. Serikat Pekerja/Serikat Buruh adalah organisasi yang dibentuk dari, ole ..."
6,PP 35/2021,"31, 32, 33, 34",20,"b. jam kesembilan, dibayar 3 (tiga) kali Upah sejam; dan c. jam kesepuluh, jam kesebelas, dan jam kedua belas, dibayar 4 (empat) kali Upah sejam. Pasal 32 (1) Perhitungan Upah Kerja Lembur didasarkan pada Upah bulanan. ( ..."
7,PP 35/2021,"34, 35, 36",21,Pasal 34 (1) Dalam hal Perusahaan telah melaksanakan pembayaran Upah Kerja Lembur dengan sebutan lain dan nilai perhitungan Upah Kerja Lembur sama dengan atau lebih baik maka perhitungan Upah Kerja Lembur tetap berlaku. ...
8,PP 35/2021,"66, 1, 2, 3, 4, 5",44,d. pelindungan Pekerja/Buruh dan perizinan berusaha pada kegiatan alih daya; e. waktu kerja pada sektor usaha atau pekerjaan tertentu; f. Waktu Kerja Lembur dan Upah Kerja Lembur; g. batasan Perusahaan tertentu yang dapa ...


## 9. Reranker Cross-Encoder → Top-3 + Relevance Score

`BAAI/bge-reranker-v2-m3` (multilingual) menilai pasangan *(pertanyaan, dokumen)* secara langsung.
Skor sudah melalui sigmoid (rentang 0–1). Hanya **Top-3** yang dikirim ke LLM.

In [24]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device=DEVICE)
TOP_K = 3
N_CANDIDATES = 10   # kandidat dari ensemble (>= 5) yang dinilai ulang reranker


def rerank(question, docs, top_k=TOP_K):
    if not docs:
        return []
    scores = reranker.predict([(question, d.page_content) for d in docs], batch_size=8, show_progress_bar=False)
    ranked = sorted(zip(docs, map(float, scores)), key=lambda pair: pair[1], reverse=True)[:top_k]
    for doc, score in ranked:
        doc.metadata["rerank_score"] = round(score, 4)
    return ranked


ranked = rerank(TEST_QUERY, ensemble_docs[:N_CANDIDATES])
show_docs([d for d, _ in ranked], [s for _, s in ranked])
print("Relevance score Top-1:", ranked[0][1])

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

,dokumen,pasal,halaman,skor,cuplikan
#,,,,,
1,PP 35/2021,"28, 30, 31",18,0.1463,(2) Perintah dan persetujuan sebagaimana dimaksud pada ayat (1) dapat dibuat dalam bentuk daftar Pekerja/Buruh yang bersedia bekerja lembur yang ditandatangani oleh Pekerja/Buruh yang bersangkutan dan Pengusaha. (3) Peng ...
2,PP 35/2021,"25, 26, 27, 28",17,0.1225,Bagian Ketiga Waktu Kerja Lembur Pasal 26 (1) Waktu Kerja Lembur hanya dapat dilakukan paling lama 4 (empat)jam dalam 1 (satu) hari dan 18 (delapan belas) jam dalam 1 (satu) minggu. (2) Ketentuan Waktu Kerja Lembur sebag ...
3,PP 35/2021,31,19,0.1115,"a. untuk ja- kerja lembur pertama sebesar 1,5 (satu koma lima) kali Upah sejam; dan b. untuk setiap ja- kerja lembur berikutnya, sebesar 2 (dua) kali Upah sejam. (2) Perusahaan yang mempekerjakan Pekerja/Buruh sebagaiman ..."


Relevance score Top-1: 0.1462651789188385


## 10. Threshold Relevance Score + Fallback DuckDuckGo

Threshold dikalibrasi dari skor Top-1 nyata: pertanyaan di dalam domain dokumen dibandingkan
pertanyaan di luar domain. Pertanyaan bahasa percakapan (test case lembur) memiliki skor jauh lebih rendah
dari pertanyaan formal, sehingga threshold harus berada di antara skor itu dan skor pertanyaan luar domain.

In [25]:
calibration_questions = {
    "Berapa lama jangka waktu maksimal PKWT?": "dalam domain (formal)",
    "Bagaimana formula perhitungan upah minimum?": "dalam domain (formal)",
    "Berapa pesangon untuk pekerja yang di-PHK karena perusahaan pailit?": "dalam domain (formal)",
    TEST_QUERY: "dalam domain (percakapan)",
    "Siapa pemenang Piala Dunia 2022?": "luar domain",
    "Bagaimana cara membuat kue brownies yang lembut?": "luar domain",
    "Berapa harga Bitcoin hari ini?": "luar domain",
}
calibration = []
for question, category in calibration_questions.items():
    top = rerank(question, ensemble_retriever.invoke(question)[:N_CANDIDATES], top_k=1)[0]
    calibration.append({"pertanyaan": question, "kategori": category, "skor Top-1": round(top[1], 4),
                        "dokumen Top-1": f"{top[0].metadata['singkatan']} hal. {top[0].metadata['page']}"})
calibration = pd.DataFrame(calibration)
display(calibration)

in_domain_min = calibration[calibration["kategori"].str.startswith("dalam")]["skor Top-1"].min()
out_domain_max = calibration[calibration["kategori"] == "luar domain"]["skor Top-1"].max()
print(f"Skor terendah dalam domain: {in_domain_min} | skor tertinggi luar domain: {out_domain_max}")

,pertanyaan,kategori,skor Top-1,dokumen Top-1
0,Berapa lama jangka waktu maksimal PKWT?,dalam domain (formal),0.9624,PP 35/2021 hal. 7
1,Bagaimana formula perhitungan upah minimum?,dalam domain (formal),0.9930,PP 51/2023 hal. 9
2,Berapa pesangon untuk pekerja yang di-PHK karena perusahaan pailit?,dalam domain (formal),0.8404,PP 35/2021 hal. 31
3,"Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?",dalam domain (percakapan),0.1463,PP 35/2021 hal. 18
4,Siapa pemenang Piala Dunia 2022?,luar domain,0.0031,UU 6/2023 hal. 2
5,Bagaimana cara membuat kue brownies yang lembut?,luar domain,0.0002,UU 6/2023 hal. 11
6,Berapa harga Bitcoin hari ini?,luar domain,0.0000,UU 6/2023 hal. 6


Skor terendah dalam domain: 0.1463 | skor tertinggi luar domain: 0.0031


In [26]:
RERANK_THRESHOLD = 0.05
if not (out_domain_max < RERANK_THRESHOLD < in_domain_min):
    # jaga-jaga bila skor di lingkungan lain bergeser: ambil titik tengah (skala log) kedua kelompok
    RERANK_THRESHOLD = round(float((max(out_domain_max, 1e-4) * in_domain_min) ** 0.5), 4)
print(f"RERANK_THRESHOLD = {RERANK_THRESHOLD}  (luar domain maks {out_domain_max} < threshold < dalam domain min {in_domain_min})")


def duckduckgo_search(query, max_results=5):
    """Cari di internet via DuckDuckGo; hasil dibungkus Document agar formatnya sama dengan dokumen lokal."""
    from ddgs import DDGS
    try:
        results = DDGS().text(query, region="id-id", safesearch="moderate", max_results=max_results)
    except Exception as error:
        print("DuckDuckGo gagal:", error)
        results = []
    return [
        Document(
            page_content=f"{r.get('title', '')}\n{r.get('body', '')}",
            metadata={"source_type": "web", "title": r.get("title", ""), "url": r.get("href", "")},
        )
        for r in results
    ]


def select_context(question, candidates, threshold=RERANK_THRESHOLD):
    """Rerank -> Top-3. Jika skor Top-1 < threshold, abaikan dokumen lokal dan pakai DuckDuckGo."""
    ranked = rerank(question, candidates)
    top1_score = ranked[0][1] if ranked else 0.0
    if top1_score >= threshold:
        return "dokumen lokal", [d for d, _ in ranked], top1_score
    return "internet (DuckDuckGo)", duckduckgo_search(question), top1_score


for question in [TEST_QUERY, "Siapa pemenang Piala Dunia 2022?"]:
    route, docs, score = select_context(question, ensemble_retriever.invoke(question)[:N_CANDIDATES])
    print(f"{question!r:85.85} | Top-1 = {score:.4f} -> {route} ({len(docs)} konteks)")

RERANK_THRESHOLD = 0.05  (luar domain maks 0.0031 < threshold < dalam domain min 0.1463)
'Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapa | Top-1 = 0.1463 -> dokumen lokal (3 konteks)
'Siapa pemenang Piala Dunia 2022?'                                                    | Top-1 = 0.0031 -> internet (DuckDuckGo) (5 konteks)


## 11. Pipeline RAG Lengkap + Sitasi

`rag_answer` merangkai semua tahap: metadata filter → HyDE → Ensemble (BM25 + semantik) → Reranker Top-3 →
threshold/DuckDuckGo → LLM fine-tuned dengan `{context}` + `{question}`. Sitasi disusun dari metadata
dokumen yang benar-benar dikirim ke LLM.

In [27]:
def rag_answer(question, metadata_filter=None, use_hyde=True):
    metadata_filter = metadata_filter or detect_metadata_filter(question)
    retriever, dense = build_hybrid_retriever(metadata_filter, use_hyde=use_hyde)
    candidates = retriever.invoke(question)[:N_CANDIDATES]
    route, context_docs, top1_score = select_context(question, candidates)

    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": RAG_PROMPT.format(context=format_context(context_docs), question=question)},
    ]
    raw_output = generate_chat(messages, max_new_tokens=640, assistant_prefix=THINK_PREFIX)[0]
    return {
        "question": question,
        "metadata_filter": metadata_filter,
        "hypotheses": getattr(dense, "last_hypotheses", []),
        "n_candidates": len(candidates),
        "route": route,
        "top1_score": top1_score,
        "context_docs": context_docs,
        "raw_output": raw_output,
        "reasoning": extract_thinking(raw_output),
        "answer": extract_final_answer(raw_output) or raw_output,
    }


def citation_rows(docs):
    rows = []
    for i, doc in enumerate(docs, start=1):
        m = doc.metadata
        if m.get("source_type") == "web":
            rows.append(f"| [{i}] | {m['title']} | - | - | [{m['url']}]({m['url']}) |")
        else:
            rows.append(f"| [{i}] | {m['singkatan']} — {m['judul']} | {m['pasal']} | {m['page']} | {m.get('rerank_score', '-')} |")
    return rows


def render_markdown(result):
    route_note = (
        f"skor reranker Top-1 = `{result['top1_score']:.4f}` "
        f"{'≥' if result['route'] == 'dokumen lokal' else '<'} threshold `{RERANK_THRESHOLD}`"
    )
    last_col = "Skor reranker" if result["route"] == "dokumen lokal" else "URL"
    sections = [
        f"### Pertanyaan\n> {result['question']}",
        f"**Sumber konteks:** {result['route']} ({route_note}) · **Filter metadata:** `{result['metadata_filter']}` "
        f"· **Kandidat ensemble:** {result['n_candidates']}",
        "#### Proses berpikir model",
        f"```text\n<think>\n{result['reasoning']}\n</think>\n```",   # code block agar tag <think> tetap terlihat
        "#### Jawaban",
        result["answer"],
        "#### Sitasi",
        "\n".join([f"| No | Dokumen | Pasal | Halaman | {last_col} |", "|---|---|---|---|---|",
                    *citation_rows(result["context_docs"])]),
    ]
    return "\n\n".join(sections)

## 12. Test Case Wajib — Model GRPO di Pipeline RAG

**Prompt:** "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"

Output harus menunjukkan proses berpikir di dalam `<think>` sebelum jawaban final berdasarkan dokumen.

In [28]:
result = rag_answer(TEST_QUERY)

print("=== Output mentah model (token spesial <think> terlihat) ===")
print(result["raw_output"])
print("\n=== Jawaban hipotetis HyDE ===")
for i, hypothesis in enumerate(result["hypotheses"], start=1):
    print(f"[{i}] {hypothesis}")

display(Markdown(render_markdown(result)))

Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=640) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Output mentah model (token spesial <think> terlihat) ===
<think>
Ketika saya mencari informasi yang relevan, saya menemukan bahwa Waktu Kerja Lembur hanya dapat dilakukan paling lama 4 jam dalam 1 hari dan 18 jam dalam 1 minggu. Jadi, 3 jam tidak termasuk dalam batas waktu kerja lembur. Selain itu, ketentuan Waktu Kerja Lembur sebagaimana dimaksud pada ayat (1) tidak termasuk kerja lembur yang dilakukan pada waktu istirahat mingguan dan/atau hari libur resmi. Oleh karena itu, saya tidak berhak mendapatkan upah lembur.
</think>
[2] PP 35/2021 tentang Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja — Pasal 26, halaman 17

=== Jawaban hipotetis HyDE ===
[1] Menurut UU No. 13 tahun 2003 tentang Ketenagakerjaan, hak untuk mendapatkan gaji lembur hanya diberikan kepada karyawan yang bekerja di luar jam kerja yang ditentukan dalam perjanjian kerja. Jika Anda tidak memiliki kesepakatan kerja yang mengizinkan lembur, Anda tidak berha

### Pertanyaan
> Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

**Sumber konteks:** dokumen lokal (skor reranker Top-1 = `0.1463` ≥ threshold `0.05`) · **Filter metadata:** `None` · **Kandidat ensemble:** 10

#### Proses berpikir model

```text
<think>
Ketika saya mencari informasi yang relevan, saya menemukan bahwa Waktu Kerja Lembur hanya dapat dilakukan paling lama 4 jam dalam 1 hari dan 18 jam dalam 1 minggu. Jadi, 3 jam tidak termasuk dalam batas waktu kerja lembur. Selain itu, ketentuan Waktu Kerja Lembur sebagaimana dimaksud pada ayat (1) tidak termasuk kerja lembur yang dilakukan pada waktu istirahat mingguan dan/atau hari libur resmi. Oleh karena itu, saya tidak berhak mendapatkan upah lembur.
</think>
```

#### Jawaban

[2] PP 35/2021 tentang Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja — Pasal 26, halaman 17

#### Sitasi

| No | Dokumen | Pasal | Halaman | Skor reranker |
|---|---|---|---|---|
| [1] | PP 35/2021 — Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja | 28, 30, 31 | 18 | 0.1463 |
| [2] | PP 35/2021 — Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja | 25, 26, 27, 28 | 17 | 0.1225 |
| [3] | PP 35/2021 — Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja | 31 | 19 | 0.1115 |

### Contoh tambahan: metadata filter otomatis & fallback internet

In [29]:
for question in [
    "Menurut PP 35/2021, berapa lama jangka waktu paling lama PKWT dan bagaimana perpanjangannya?",
    "Siapa pemenang Piala Dunia 2022?",
]:
    display(Markdown(render_markdown(rag_answer(question))))
    display(Markdown("---"))

Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=640) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Pertanyaan
> Menurut PP 35/2021, berapa lama jangka waktu paling lama PKWT dan bagaimana perpanjangannya?

**Sumber konteks:** dokumen lokal (skor reranker Top-1 = `0.9701` ≥ threshold `0.05`) · **Filter metadata:** `{'peraturan_id': 'PP-35-2021'}` · **Kandidat ensemble:** 10

#### Proses berpikir model

```text
<think>
Menurut PP 35/2021, jangka waktu paling lama PKWT adalah 5 tahun. Perpanjangan PKWT hanya dapat dilakukan dengan kesepakatan antara Pengusaha dan Pekerja/Buruh, dengan jangka waktu keseluruhan PKWT beserta perpanjangannya tidak lebih dari 5 tahun.
</think>
```

#### Jawaban

[1] PP 35/2021 tentang Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja.

#### Sitasi

| No | Dokumen | Pasal | Halaman | Skor reranker |
|---|---|---|---|---|
| [1] | PP 35/2021 — Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja | 6, 7, 8 | 7 | 0.9701 |
| [2] | PP 35/2021 — Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja | 14, 15, 16 | 11 | 0.9399 |
| [3] | PP 35/2021 — Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja | 66 | 43 | 0.897 |

---

Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=640) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Pertanyaan
> Siapa pemenang Piala Dunia 2022?

**Sumber konteks:** internet (DuckDuckGo) (skor reranker Top-1 = `0.0031` < threshold `0.05`) · **Filter metadata:** `None` · **Kandidat ensemble:** 10

#### Proses berpikir model

```text
<think>
Kutip kata demi kata kalimat pasal yang relevan beserta nomor sumbernya, misalnya: [5] Inilah Daftar Lengkap Pemenang Penghargaan Piala Dunia 2022 - Inilah Daftar Lengkap Pemenang Penghargaan Piala Dunia 2022.
</think>
```

#### Jawaban

Jawaban akhir: Timnas Argentina resmi menjadi juara Piala Dunia 2022 di Qatar setelah mengalahkan Prancis pada final di Stadion Lusail, Minggu (18/12/2022). Kedua tim bermain 3-3 hingga babak perpanjangan waktu dan Argentina menang adu penalti 4-2. Kemenangan Argentina semakin lengkap dengan tiga pemainnya yang membawa pulang hadiah individu di turnamen paling bergengsi tersebut. [5] Inilah Daftar Lengkap Pemenang Penghargaan Piala Dunia 2022.

#### Sitasi

| No | Dokumen | Pasal | Halaman | URL |
|---|---|---|---|---|
| [1] | Final Piala Dunia FIFA 2022 - Wikipedia bahasa Indonesia, ensiklopedia ... | - | - | [https://id.wikipedia.org/wiki/Final_Piala_Dunia_FIFA_2022](https://id.wikipedia.org/wiki/Final_Piala_Dunia_FIFA_2022) |
| [2] | Piala Dunia FIFA 2022 - Wikipedia bahasa Indonesia, ensiklopedia bebas | - | - | [https://id.wikipedia.org/wiki/Piala_Dunia_FIFA_2022](https://id.wikipedia.org/wiki/Piala_Dunia_FIFA_2022) |
| [3] | Pentas Akhir Piala Dunia FIFA 2022 - Wikipedia Bahasa Melayu ... | - | - | [https://ms.wikipedia.org/wiki/Pentas_Akhir_Piala_Dunia_FIFA_2022](https://ms.wikipedia.org/wiki/Pentas_Akhir_Piala_Dunia_FIFA_2022) |
| [4] | Piala Dunia FIFA 2022 - Wikipedia Bahasa Melayu, ensiklopedia bebas | - | - | [https://ms.wikipedia.org/wiki/Piala_Dunia_FIFA_2022](https://ms.wikipedia.org/wiki/Piala_Dunia_FIFA_2022) |
| [5] | Inilah Daftar Lengkap Pemenang Penghargaan Piala Dunia 2022 | - | - | [https://www.inilah.com/inilah-daftar-lengkap-pemenang-penghargaan-piala-dunia-2022](https://www.inilah.com/inilah-daftar-lengkap-pemenang-penghargaan-piala-dunia-2022) |

---

## 13. Interface: Interactive Python Loop

Ketik pertanyaan lalu tekan Enter. Ketik `exit` untuk keluar. Jawaban dicetak dengan `IPython.display.Markdown`.

In [30]:
EXIT_COMMANDS = {"exit", "quit", "keluar", "selesai"}


def legal_assistant_loop():
    display(Markdown("## Asisten Hukum Legal Team\nKetik pertanyaan tentang UU 6/2023, PP 35/2021, PP 51/2023, "
                     "atau PP 5/2021. Ketik `exit` untuk keluar."))
    while True:
        try:
            question = input("Pertanyaan Anda: ").strip()
        except (EOFError, KeyboardInterrupt, StdinNotImplementedError):
            display(Markdown("_Input interaktif tidak tersedia di lingkungan ini. Sesi diakhiri._"))
            break
        if not question:
            continue
        if question.lower() in EXIT_COMMANDS:
            display(Markdown("_Sesi selesai. Terima kasih._"))
            break
        display(Markdown(render_markdown(rag_answer(question))))
        display(Markdown("---"))


legal_assistant_loop()

## Asisten Hukum Legal Team
Ketik pertanyaan tentang UU 6/2023, PP 35/2021, PP 51/2023, atau PP 5/2021. Ketik `exit` untuk keluar.

_Input interaktif tidak tersedia di lingkungan ini. Sesi diakhiri._

## Kesimpulan

- Empat PDF wajib dimuat, dibersihkan dari noise OCR, diperkaya metadata, lalu dipotong menjadi parent (2000/200)
  dan child chunk (400/80).
- Embedding open-source `BAAI/bge-m3` disimpan di ChromaDB lokal; Parent-Child Retriever mengembalikan parent chunk.
- Ensemble BM25 + semantik (bobot 0.4/0.6) menghasilkan ≥ 5 kandidat; metadata filter otomatis membatasi pencarian
  bila pertanyaan menyebut peraturan tertentu.
- HyDE (2 jawaban hipotetis) memperbaiki pencarian untuk pertanyaan bahasa percakapan.
- Reranker `bge-reranker-v2-m3` memilih Top-3; skor Top-1 di bawah threshold mengalihkan ke DuckDuckGo.
- Generator adalah model fine-tuned Kriteria 1 (SFT → GRPO) yang menampilkan `<think>` sebelum jawaban dan sitasi.